In [2]:
from IPython.display import Image

- 模态鸿沟
- 文本是一维的，图像是二维的，视频就是三维的（也许是四维的）
    - 文本：text sequence
    - img：height * width
    - video：temporal of img seq，& audio

## 什么是原生多模态

在 AI 领域，“原生多模态”通常指的是模型并非由独立的单模态模型（如纯文本 LLM + 纯视觉编码器）通过简单的“胶水层”拼接而成，而是从模型设计之初或预训练的一开始，就将多种模态（文本、图像、视频、音频）作为平等的 Token 输入到同一个 Transformer 网络中进行端到端的联合训练。

原生多模态的核心特征：

- 单一模型架构： 不区分明显的“视觉编码器”和“语言解码器”，或者两者融合得极其紧密。
- 从头联合训练： 模型在预训练初期就接触多模态数据，而不是先训练好一个强的 LLM 再去“看”图。
- 输入输出任意： 能够像 Gemini 那样实现 Any-to-Any（如输入视频直接输出语音，中间不转文本）。

### Gemini

- Gemini models support interleaved sequences of text, image, audio, and video as inputs (illustrated by tokens of different colors in the input sequence). They can output responses with interleaved image and text.
    - 没有分阶段：Gemini不是先训练一个纯文本的LLM，再用多模态数据进行第二阶段的微调。而是从预训练的第一天开始，就在同时学习所有模态。
    - 统一的Transformer核心：如图2所示，所有模态的输入（文本token、图像帧、音频特征）都被转换成一个统一的向量序列，然后被送入同一个Transformer解码器进行处理。
    - 联合训练带来的深远影响
        - 统一的内部表示空间 (Unified Representation Space)：由于从头开始联合训练，Gemini的神经网络权重被迫去学习不同模态之间的深层关联。它不是为图像信息在已有的文本世界观里“找个位置”，而是构建了一个统一的、可以同时容纳和关联视觉、听觉、语言概念的内部世界。
        - 实现真正的跨模态推理 (Deep Cross-modal Reasoning)：因为底层表示是统一的，Gemini可以进行更复杂的推理。
- Initial pretraining used **random initialization**. Post-training was initialized from checkpoints obtained at the later stages of pretraining. These checkpoints were fine-tuned using supervised fine-tuning, and subsequently used to initialize reward model training and RLHF.
- Gemini is trained jointly across image, audio, video and text data for the purpose of building a model with both strong generalist capabilities across modalities alongside cutting-edge understanding and reasoning performance in each respective domain.

In [3]:
Image(url='./figs/gemini-multi-modal.png', width=800)

### qwen3-vl

Qwen3-VL 属于“组合式架构”（Compositional Architecture），而非通过单一 Transformer 从零训练的“原生”模型；但从能力表现和训练深度上看，它实现了“原生级”的融合体验。Qwen3-VL 采用了经典的三模块架构（Qwen3-VL adopts a three-module architecture comprising a vision encoder, an MLP-based vision–language merger, and a large language model (LLM).）：
- 视觉编码器 (Vision Encoder)： 基于 SigLIP-2。
    - We utilize the SigLIP-2 architecture
    - NaViT: native resolution
- 视觉-语言融合层 (Merger)： 基于 MLP（多层感知机）。
    - As in Qwen2.5-VL, we use a two-layer MLP to compress 2 × 2 visual features from the vision encoder into a single visual token, aligned with the LLM’s hidden dimension.
    - 16 * 16 => 32 * 32
    - https://huggingface.co/Qwen/Qwen3-VL-30B-A3B-Instruct/blob/main/config.json
- 大语言模型 (LLM)： 基于 Qwen3。

虽然架构是组合的，但报告中提到它“原生支持交错上下文 (natively supports interleaved contexts)” 。这里的“原生”更多是指功能层面——它能够像处理文本一样自然地处理图文混排、视频流，且支持 256K 超长上下文（It natively supports interleaved contexts of up to 256K tokens, seamlessly integrating text, images, and video.）。

Qwen3-VL 的团队明确提出了一种**“分而治之 (divide-and-conquer)”** 的策略。（Multimodal reasoning lies at the heart of Qwen3-VL, with STEM reasoning constituting its most essential part. Our philosophy follows a divide-and-conquer strategy: we first develop fine-grained visual perception and robust linguistic reasoning capabilities **independently**, and then integrate them in a synergistic manner to achieve effective multimodal reasoning.）

- 先分别开发最强的视觉感知能力（SigLIP-2）和语言推理能力（Qwen3）。
- 然后再通过协同的方式将它们集成 。 这与“从零开始就在一起训练”的原生路径在理念上有本质区别。

Qwen3-VL 意识到了“组合式架构”的潜在弱点：如果只把视觉编码器的最终输出给 LLM，可能会丢失细节。（不同于以往将所有视觉 token 仅输入到 LLM 输入层的做法，Qwen3-VL 的 DeepStack 抽取视觉编码器（ViT）的中间层特征，并将其注入到 LLM 的特定层中。）
- 解决方案 (DeepStack): 它不只是“看”一眼（only input），而是把视觉编码器不同层级（从浅层纹理到深层语义）的特征都提取出来，注入到 LLM 的不同层中。
    - DeepStack 采用的是**残差连接（Residual Connections）**的方式 。
    - 它将视觉特征直接“加”到 LLM 内部隐藏状态（Hidden States）上，而不是拼接到序列的开头或结尾。因此，对于 LLM 来说，处理的 Seq Length 保持不变

- visual encoding
    - The visual encoding of Gemini models is inspired by our own foundational work on **Flamingo, CoCa, and PaLI**(2022), with the important distinction that the models are multimodal from the beginning and can natively output images using discrete image tokens.
- Flamingo：利用现成的强大视觉模型和语言模型，通过“插入式”的可训练层将二者连接，实现强大的少样本（Few-shot）学习能力。
    - 视觉端： 使用预训练且冻结的 NFNet (Normalizer-Free ResNet)。它不输出单一向量，而是输出 2D 空间特征图（Spatial Grid Features）。
    - **Perceiver Resampler**（核心创新）： 这是一个可训练的模块。它接收视觉端输出的数量不定的特征图，并将它们“重采样”为固定数量（例如 64 个）的视觉 Token。这解决了不同分辨率或长视频导致视觉输入过长的问题，强制将视觉信息压缩成高密度的摘要。
    - 语言端： 使用预训练且冻结的 Chinchilla 模型（Decoder-only LLM）。
    - Gated Cross-Attention (门控交叉注意力)： Flamingo 在冻结的 LLM 层之间插入了新的 Gated Cross-Attention - Dense 层。
  - 冻结策略： 视觉编码器和 LLM 权重完全冻结，只训练 Perceiver Resampler 和插入的交叉注意力层。
  - 数据： 关键在于使用了 Interleaved（交错）数据（如 M3W 数据集），即“文本-图像-文本”穿插的网页数据，而不仅仅是成对的（image, text）数据。这赋予了 Flamingo 上下文内学习（In-context Learning）的能力。
- PaLI (Pathways Language and Image model):
    - 视觉端： 使用巨型的 ViT (如 ViT-e, ViT-G)。图像被切分为 Patch，通过 ViT 变成序列化的 Visual Tokens。
    - 文本端： 基于 mT5 或 UL2 的 Encoder-Decoder 架构。
    - 直接作为输入： 与 Flamingo 的“侧面插入”不同，PaLI 将 ViT 输出的 Visual Tokens 直接投影并拼接到文本 Token 序列的前面，作为 Encoder 的输入。
    - 复用与继续训练： PaLI 复用了预训练好的大型 ViT 和 mT5 的权重，但不同于 Flamingo 的冻结策略，PaLI 对这些权重进行联合微调/继续训练。这允许视觉和语言模块深度融合。
    - 任务混合 (Task Mixture)： 训练任务极其丰富，包括 Span Corruption（填空）、Captioning、VQA、OCR 等。
    - 多语言 (Multilingual)： 继承了 mT5 的特性，PaLI 在 100 多种语言上进行了训练，强调了多模态在多语言环境下的通用性。

### qwen3-vl 多阶段训练

Qwen3-VL 采用了**“先部分冻结，后全参数解冻”**的策略，具体分为以下阶段：

- Stage 0: 视觉-语言对齐 (Vision-Language Alignment)
    - 这是初始的“热身”阶段，旨在快速打通视觉与语言的模态连接。
    - 训练 (Trainable): 仅 视觉-语言融合层 (Merger)。
    - 冻结 (Frozen): 视觉编码器 (Vision Encoder) 和 大语言模型 (LLM)。
    - 目的： 在不破坏预训练好的视觉和语言特征的前提下，训练“胶水层”让 LLM 能看懂视觉信号。
- Stage 1: 多模态预训练 (Multimodal Pre-Training) 这是最关键的全面训练阶段，不再有任何冻结。
    - 训练 (Trainable): 所有组件（视觉编码器、融合层、LLM）全部解冻，进行端到端的联合训练。
    - 冻结 (Frozen): 无。
    - 目的： 让视觉编码器适应更复杂的任务，同时让 LLM 深度理解视觉信息。
- Stage 2 & 3: 长上下文训练 (Long-Context Pre-Training & Adaptation)：后续的长文本/长视频能力扩展阶段，延续了全参数训练的模式。
    - 训练 (Trainable): 所有组件继续保持可训练状态。
    - 冻结 (Frozen): 无。
    - 目的： 在 Stage 2 将上下文扩展至 32k，在 Stage 3 进一步扩展至 256k，全参数微调以确保长距离依赖的学习效果。
| Stage | Objective (训练目标) | Trainable Components (训练组件) | Token Budget (数据量) | Seq Length (序列长度) | Key Inputs & Data (主要输入与数据) | Task & Outputs (任务与输出) |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Stage 0 (S0)** | **Vision-Language Alignment**<br>(视觉-语言对齐) | **Merger Only**<br>(仅融合层)<br>*Vision & LLM Frozen* | ~67B | 8,192 | **Inputs:**<br>- High-quality Image-Caption pairs<br>- Visual knowledge collections<br>- OCR data | **Task:**<br>- Image Captioning<br>- Text Recognition<br>**Output:**<br>- Basic descriptive text<br>- OCR text results |
| **Stage 1 (S1)** | **Multimodal Pre-Training**<br>(多模态预训练) | **All Components**<br>(全参数解冻，训练)<br>*Vision + Merger + LLM* | ~1T | 8,192 | **Inputs:**<br>- Interleaved Image-Text docs<br>- Visual Grounding data<br>- VQA datasets<br>- STEM & Video data<br>- Text-only corpus | **Task:**<br>- General VQA<br>- Visual Grounding (Box/Point)<br>- Reasoning (STEM)<br>**Output:**<br>- Answers to queries<br>- Bounding box coordinates<br>- Reasoning steps |
| **Stage 2 (S2)** | **Long-Context Pre-Training**<br>(长上下文预训练) | **All Components**<br>(全参数训练) | ~1T | 32,768(32k) | **Inputs:**<br>- More Text-only data (for long-form)<br>- Video data (Time-aware)<br>- Agent instruction data | **Task:**<br>- Long-form text understanding<br>- Video understanding<br>- Multi-step agent instructions<br>**Output:**<br>- Coherent long text<br>- Temporal localization |
| **Stage 3 (S3)** | **Ultra-Long-Context Adaptation**<br>(超长上下文适配) | **All Components**<br>(全参数训练) | 100B | 262,144(256k) | **Inputs:**<br>- Long-video (up to 2h+)<br>- Long-document (Book-scale)<br>- Interleaved sequences | **Task:**<br>- Needle-in-a-Haystack retrieval<br>- Long video summarization<br>- Document analysis<br>**Output:**<br>- Extraction from massive context |

## AIGC

### history

- ImageGPT & ViT & Swin Transformer
    - imagegpt: pixel-level tokenize, decoder-only Transformer，自回归预测下一个 pixel
    - ViT: patch-level tokenize, encoder-only Transformer, 主要聚焦图像分类任务；
    - Swin Transformer: 分层 encoder，更广泛的图像任务

### modern

- https://www.zhihu.com/question/1908479621466396378/answer/1910745238147929889
    - diffusion和自回归的最本质区别在于：过去生成完的东西能不能改，能改则是 diffusion，不能改则是 ar；
- 图像生成领域
    - ar：vqgan，maskGiT
        - 建模 $p(x)=p(x_1,\cdots,x_n)=p(x_1)p(x_2|x_1)\cdots p(x_n|x_{\lt {n-1}})$
        - $x_1,x_2,\cdots,x_{n-1}$: 这些历史 tokens，一旦确定下来，就不会被修改了，生成 eos 的时机也变得不确定；
    - diffusion：DDPM，flow matching
        - 建模的则是给定噪声分布，转移到联合分布：$p(x_1,\cdots,x_n|\epsilon)$，为了获得比较好的效果，
        - 从噪声到信号分好几步，$p(x_1^t,\cdots,x_n^{t}|x_1^{t-1},\cdots,x_n^{t-1})$
        - 中间的每一步，$x$ 都是被修改的，但 $x$ 的shape（即长度）是事先要定下来的；
- video generation 领域，以主要从 stable diffusion => flow matching